# DORA ICT Incident Classification & Vendor Compliance Demo

**DORA** (Digital Operational Resilience Act) requires EU financial institutions to classify ICT incidents, report within 4 hours, and maintain vendor contractual controls — and **prove compliance on demand**.

This notebook demonstrates factpy as a cross-domain auditable reasoning framework:
- 8 DORA rules encoded as formal Datalog
- 3 ICT incidents + 2 third-party vendors
- Full pipeline: evidence tree → certainty → provenance → audit → static HTML

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

import json, tempfile, shutil
from pprint import pprint

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field, Rule, Pred, vars as sdk_vars
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation,
    explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    export_runtime_package, accept_runtime_derivation,
)
from factpy_kernel.audit import AuditQuery, load_audit_package
from factpy_kernel.audit.static_ui import render_audit_static_site

print("Imports OK")

## 2. Schema

- **ICTIncident**: affected clients, financial impact, duration, reporting status
- **DORAThreshold**: regulatory classification thresholds
- **ICTVendor**: contractual compliance attributes

In [ ]:
class ICTIncident(Entity):
    incident_id: str = Identity(primary_key=True)
    locale: str = Identity()
    description: str = Field(cardinality="single")
    affected_clients: str = Field(cardinality="single")
    financial_impact_eur: str = Field(cardinality="single")
    duration_hours: str = Field(cardinality="single")
    reporting_status: str = Field(cardinality="single")

class DORAThreshold(Entity):
    threshold_id: str = Identity(primary_key=True)
    locale: str = Identity()
    client_threshold: str = Field(cardinality="single")
    financial_threshold_eur: str = Field(cardinality="single")
    duration_threshold_hours: str = Field(cardinality="single")

class ICTVendor(Entity):
    vendor_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    has_audit_rights: str = Field(cardinality="single")
    has_exit_strategy: str = Field(cardinality="single")
    has_subcontracting_controls: str = Field(cardinality="single")

sdk = SDKStore([ICTIncident, DORAThreshold, ICTVendor])
print(f"Schema: {len(sdk.schema_ir['predicates'])} predicates")

## 3. Rules (8 DORA rules)

**Incident**: 3 threshold breaches + major classification (OR) + reporting ok/late
**Vendor**: compliant (all clauses) + non-compliant (missing exit strategy)

In [ ]:
with sdk_vars("inc","thr","vendor","clients","amount","hours",
             "client_thr","amount_thr","duration_thr",
             "status","audit_rights","exit_strategy","subcontracting") as (
    inc,thr,vendor,clients,amount,hours,
    client_thr,amount_thr,duration_thr,
    status,audit_rights,exit_strategy,subcontracting):

    client_breach = Rule(id="q.dora_client_breach", version="1.0.0", select=[inc,clients],
        where=[Pred("ict_incident:affected_clients",inc,clients),
               Pred("dora_threshold:client_threshold",thr,client_thr),
               clients >= client_thr], expose=True,
        condition_weights={"b0.a0": 0.9, "b0.a1": 0.3})

    financial_breach = Rule(id="q.dora_financial_breach", version="1.0.0", select=[inc,amount],
        where=[Pred("ict_incident:financial_impact_eur",inc,amount),
               Pred("dora_threshold:financial_threshold_eur",thr,amount_thr),
               amount >= amount_thr], expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.4})

    duration_breach = Rule(id="q.dora_duration_breach", version="1.0.0", select=[inc,hours],
        where=[Pred("ict_incident:duration_hours",inc,hours),
               Pred("dora_threshold:duration_threshold_hours",thr,duration_thr),
               hours >= duration_thr], expose=True,
        condition_weights={"b0.a0": 0.7, "b0.a1": 0.5})

    major_incident = Rule(id="q.dora_major_incident", version="1.0.0", select=[inc,status],
        where=[[RuleRef(client_breach)(inc,clients), status == "major"],
               [RuleRef(financial_breach)(inc,amount), status == "major"],
               [RuleRef(duration_breach)(inc,hours), status == "major"]], expose=True)

    reporting_compliant = Rule(id="q.dora_reporting_compliant", version="1.0.0", select=[inc,status],
        where=[Pred("ict_incident:reporting_status",inc,status), status == "reported_within_4h"],
        expose=True, condition_weights={"b0.a0": 1.0})

    reporting_noncompliant = Rule(id="q.dora_reporting_noncompliant", version="1.0.0", select=[inc,status],
        where=[Pred("ict_incident:reporting_status",inc,status), status == "late_report"],
        expose=True, condition_weights={"b0.a0": 1.0})

    vendor_compliant = Rule(id="q.dora_vendor_compliant", version="1.0.0", select=[vendor,status],
        where=[Pred("ict_vendor:has_audit_rights",vendor,audit_rights), audit_rights == "yes",
               Pred("ict_vendor:has_exit_strategy",vendor,exit_strategy), exit_strategy == "yes",
               Pred("ict_vendor:has_subcontracting_controls",vendor,subcontracting), subcontracting == "yes",
               status == "compliant"], expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a2": 0.9, "b0.a4": 0.7})

    vendor_noncompliant = Rule(id="q.dora_vendor_noncompliant_no_exit", version="1.0.0", select=[vendor,status],
        where=[Pred("ict_vendor:has_exit_strategy",vendor,exit_strategy), exit_strategy == "no",
               status == "non_compliant_missing_exit_strategy"], expose=True,
        condition_weights={"b0.a0": 1.0})

all_rules = [client_breach, financial_breach, duration_breach, major_incident,
             reporting_compliant, reporting_noncompliant, vendor_compliant, vendor_noncompliant]
print(f"{len(all_rules)} rules defined")
for r in all_rules: print(f"  {r.id}")

## 4. Registry + Data

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="dora_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
for rule in all_rules:
    registry.register_rule_spec(sdk._compile_rule_input(rule))

with sdk.batch() as tx:
    for iid, desc, cl, fi, dur, rep in [
        ("INC-042","Payment gateway outage","15000","2300000","6","reported_within_4h"),
        ("INC-043","Core banking degradation","45000","5100000","12","late_report"),
        ("INC-044","Minor email delay","200","5000","1","reported_within_4h")]:
        e = tx.entity(ICTIncident, incident_id=iid, locale="en")
        e.description.set(desc); e.affected_clients.set(cl)
        e.financial_impact_eur.set(fi); e.duration_hours.set(dur)
        e.reporting_status.set(rep)

    t = tx.entity(DORAThreshold, threshold_id="DORA-2025", locale="en")
    t.client_threshold.set("10000"); t.financial_threshold_eur.set("1000000")
    t.duration_threshold_hours.set("4")

    for vid, nm, ar, es, sc in [
        ("VENDOR-A","Acme Cloud Services","yes","yes","yes"),
        ("VENDOR-B","QuickPay Gateway","yes","no","yes")]:
        v = tx.entity(ICTVendor, vendor_id=vid, locale="en")
        v.name.set(nm); v.has_audit_rights.set(ar)
        v.has_exit_strategy.set(es); v.has_subcontracting_controls.set(sc)
    tx.commit()

print("Registry + data ready")

## 5. Runtime Session + Evaluate

In [ ]:
reset_runtime_sessions_for_tests()
sess = open_runtime_session({"registry_root": registry_dir})
session_id = sess["session"]["session_id"]

for iid, fields in [
    ("INC-042",[("affected_clients","15000"),("financial_impact_eur","2300000"),
                ("duration_hours","6"),("reporting_status","reported_within_4h")]),
    ("INC-043",[("affected_clients","45000"),("financial_impact_eur","5100000"),
                ("duration_hours","12"),("reporting_status","late_report")]),
    ("INC-044",[("affected_clients","200"),("financial_impact_eur","5000"),
                ("duration_hours","1"),("reporting_status","reported_within_4h")])]:
    ref = sdk.ref(ICTIncident, incident_id=iid, locale="en")
    for s,v in fields:
        write_runtime_fact(session_id, {"pred_id":f"ict_incident:{s}","e_ref":ref,
            "rest_terms":[["string",v]],"meta":{"confidence":0.92} if s=="affected_clients" else {}}, kind="add")

thr_ref = sdk.ref(DORAThreshold, threshold_id="DORA-2025", locale="en")
for s,v in [("client_threshold","10000"),("financial_threshold_eur","1000000"),("duration_threshold_hours","4")]:
    write_runtime_fact(session_id, {"pred_id":f"dora_threshold:{s}","e_ref":thr_ref,"rest_terms":[["string",v]]}, kind="add")

for vid, clauses in [("VENDOR-A",{"has_audit_rights":"yes","has_exit_strategy":"yes","has_subcontracting_controls":"yes"}),
                      ("VENDOR-B",{"has_audit_rights":"yes","has_exit_strategy":"no","has_subcontracting_controls":"yes"})]:
    ref = sdk.ref(ICTVendor, vendor_id=vid, locale="en")
    for s,v in clauses.items():
        write_runtime_fact(session_id, {"pred_id":f"ict_vendor:{s}","e_ref":ref,"rest_terms":[["string",v]]}, kind="add")

# Evaluate all derivations
results = {}
for drv_id, rule_id, target, v1, v2, label in [
    ("drv.major","q.dora_major_incident","ict_incident:reporting_status","$inc","$status","Major Incident"),
    ("drv.report_ok","q.dora_reporting_compliant","ict_incident:reporting_status","$inc","$status","Reporting OK"),
    ("drv.report_late","q.dora_reporting_noncompliant","ict_incident:reporting_status","$inc","$status","Reporting Late"),
    ("drv.vendor_ok","q.dora_vendor_compliant","ict_vendor:has_audit_rights","$vendor","$status","Vendor Compliant"),
    ("drv.vendor_bad","q.dora_vendor_noncompliant_no_exit","ict_vendor:has_exit_strategy","$vendor","$status","Vendor Non-Compliant")]:
    ev = evaluate_runtime_derivation(session_id, {"derivation":{
        "derivation_id":drv_id,"version":"1.0.0","target":target,
        "head_vars":[v1,v2],"where":[["ruleref",rule_id,"1.0.0",[v1,v2]]],"mode":"native"}})
    results[label] = ev
    n = len(ev["evaluation"]["candidates"]) if ev["ok"] else 0
    print(f"{label}: {n} candidates")

print("\nAll evaluations complete")

## 6. Results: Incident Classification + Vendor Compliance

In [ ]:
for label, ev in results.items():
    print(f"\n{label}:")
    if ev["ok"]:
        for c in ev["evaluation"]["candidates"]:
            t = c["payload"]["terms"]
            ck = c.get("confidence_kind","none")
            entity = t[0]["value"].split(":")[-1][:25]
            print(f"  {entity}... -> {t[1]['value']}  (confidence_kind={ck})")
            if ck == "certainty":
                sm = explain_runtime_summary(session_id, {"kind":"candidate","id":c["candidate_id"]})
                if sm["ok"] and sm.get("certainty_summary"):
                    cs = sm["certainty_summary"]
                    print(f"    Certainty: {cs['aggregate_certainty']} ({cs['aggregation']})")

## 7. Audit: Accept + Export + Static Site

In [ ]:
for ev in results.values():
    if ev["ok"]:
        for cand in ev["evaluation"]["candidates"]:
            accept_runtime_derivation(session_id, {"candidate": cand})

audit_dir = tempfile.mkdtemp(prefix="dora_audit_")
export_runtime_package(session_id, {"out_dir": audit_dir, "package_kind": "audit"})
pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)
print(f"Audit candidates: {len(aq.list_candidates())}")

site_dir = tempfile.mkdtemp(prefix="dora_site_")
render_audit_static_site(audit_dir, site_dir)
pages = list(Path(site_dir).rglob("*.html"))
print(f"Static site: {len(pages)} pages")
print(f"Open: {site_dir}/index.html")

## 8. Cleanup

In [ ]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print("DORA demo complete. Same framework, ECSS + DORA.")

## Summary

| Check | INC-042 (15K clients, €2.3M, 6h) | INC-043 (45K clients, €5.1M, 12h) | INC-044 (200 clients, €5K, 1h) |
|-------|---|---|---|
| Major? | ✅ YES | ✅ YES | ❌ NO |
| Reported on time? | ✅ | ❌ late | ✅ |

| Vendor | Audit Rights | Exit Strategy | Subcontracting | Compliant? |
|--------|---|---|---|---|
| Acme Cloud | ✅ | ✅ | ✅ | ✅ YES |
| QuickPay | ✅ | ❌ | ✅ | ❌ NO |

**Cross-domain validated**: same auditable reasoning framework handles ECSS (space) + DORA (finance).